# Theoretical H₂ consumption from sulfate depletion

For each sulfate dataset we compute two things:

**1. H₂ from the sulfate actually consumed** (what really happened):

delta_sulfate = (sulfate_start − sulfate_end) × V_L

n_H2 = 4 × delta_sulfate

**2. Theoretical maximum H₂** if *all* the starting sulfate were reduced to zero
(i.e. the most H₂ the microbes could ever consume given the sulfate available at the start):

delta_sulfate_max = sulfate_start × V_L

n_H2_max = 4 × delta_sulfate_max

Sulfate values are concentrations in **mM (mmol/L)** and the volume is in **L**, so
`mM × L = mmol` → results are in **mmol**.

**Run the cells top to bottom.**

## 1. Install dependencies (into this notebook's kernel)
`%pip` installs into the exact Python running this notebook, which avoids the
"already satisfied" / `ModuleNotFoundError` mismatch you hit in the terminal.

## 2. Settings

In [ ]:
import pandas as pd

# Path to the rearranged Excel file
INPUT_FILE  = ""
OUTPUT_FILE = ""

# ============================================================
# Reactor liquid volume in LITERS.  Measured volume = 67.5 +/- 2 mL.
VOLUME_L = 0.0675       # 67.5 mL
# ============================================================

H2_PER_SULFATE = 4      # mol H2 consumed per mol sulfate

## 3. Load the data

In [6]:
df = pd.read_excel(INPUT_FILE)
df.head()

,ID,run,cyl,p_bar,medium,date_start,date_end,day_start,day_end,OD600_start,...,acetate_mM_run_start,acetate_mM_run_end,sulfate_mM_run_start,sulfate_mM_run_end,acetate_mM_run1_start,acetate_mM_run1_end,sulfate_mM_run1_start,sulfate_mM_run1_end,Unnamed: 21,prefix
0,run3_cyl1,3,1,100,NORCE,2025-06-18,2025-06-27,0.0,9.0,0.061,...,NaN,NaN,NaN,NaN,2.232385,1.862297,23.031959,9.629919,NaN,2025-06-18_h2-100bar_SRB
1,run3_cyl2,3,2,100,NORCE,2025-06-18,2025-06-27,0.0,9.0,0.005,...,NaN,NaN,NaN,NaN,0.700373,0.686822,23.020508,25.069228,NaN,NaN
2,run4_cyl1,4,1,100,NORCE,2025-07-01,2025-07-15,0.0,14.0,0.079,...,2.355691,1.741192,22.581303,9.765563,1.957995,1.618394,22.701957,10.603789,NaN,NaN
3,run4_cyl2,4,2,100,NORCE,2025-07-01,2025-07-15,0.0,14.0,0.009,...,0.470190,0.451220,23.720175,23.675619,0.697832,0.684282,23.982927,23.307828,NaN,NaN
4,run5_cyl1,5,1,60,NORCE,2025-10-23,2025-11-10,0.0,18.0,0.051,...,2.145664,3.953930,22.359775,8.389756,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Calculate ΔSO₄ and H₂

In [7]:
def consumed_hydrogen(start, end, volume_l):
    """Returns (delta_sulfate, n_hydrogen, delta_sulfate_max, n_hydrogen_max) in mmol.

    delta_sulfate / n_hydrogen      -> from the sulfate ACTUALLY consumed (start - end)
    delta_sulfate_max / n_hydrogen_max -> theoretical max if ALL starting sulfate were reduced
    Missing values give NaN for whatever cannot be computed.
    """
    # theoretical max only needs the start value
    if pd.isna(start):
        delta_max = float("nan")
        h_max     = float("nan")
    else:
        delta_max = start * volume_l               # mmol if all sulfate consumed
        h_max     = H2_PER_SULFATE * delta_max

    # actual consumption needs both start and end
    if pd.isna(start) or pd.isna(end):
        return float("nan"), float("nan"), delta_max, h_max
    delta = (start - end) * volume_l               # mmol consumed
    return delta, H2_PER_SULFATE * delta, delta_max, h_max

# start from the full rearranged table, then append the computed columns
res = df.copy()

calc = []
for _, r in df.iterrows():
    d_run,  h_run,  dmax_run,  hmax_run  = consumed_hydrogen(r.get("sulfate_mM_run_start"),
                                                             r.get("sulfate_mM_run_end"),  VOLUME_L)
    d_run1, h_run1, dmax_run1, hmax_run1 = consumed_hydrogen(r.get("sulfate_mM_run1_start"),
                                                             r.get("sulfate_mM_run1_end"), VOLUME_L)
    calc.append({
        "delta_sulfate_run_mmol":       d_run,
        "n_hydrogen_run_mmol":          h_run,
        "n_hydrogen_run_max_mmol":      hmax_run,
        "delta_sulfate_run1_mmol":      d_run1,
        "n_hydrogen_run1_mmol":         h_run1,
        "n_hydrogen_run1_max_mmol":     hmax_run1,
        "both_sulfate_data":            pd.notna(h_run) and pd.notna(h_run1),
    })

res = pd.concat([res, pd.DataFrame(calc, index=res.index)], axis=1)
res

,ID,run,cyl,p_bar,medium,date_start,date_end,day_start,day_end,OD600_start,...,sulfate_mM_run1_end,Unnamed: 21,prefix,delta_sulfate_run_mmol,n_hydrogen_run_mmol,n_hydrogen_run_max_mmol,delta_sulfate_run1_mmol,n_hydrogen_run1_mmol,n_hydrogen_run1_max_mmol,both_sulfate_data
0,run3_cyl1,3,1,100,NORCE,2025-06-18,2025-06-27,0.0,9.0,0.061,...,9.629919,NaN,2025-06-18_h2-100bar_SRB,NaN,NaN,NaN,0.904638,3.618551,6.218629,False
1,run3_cyl2,3,2,100,NORCE,2025-06-18,2025-06-27,0.0,9.0,0.005,...,25.069228,NaN,NaN,NaN,NaN,NaN,-0.138289,-0.553154,6.215537,False
2,run4_cyl1,4,1,100,NORCE,2025-07-01,2025-07-15,0.0,14.0,0.079,...,10.603789,NaN,NaN,0.865062,3.460250,6.096952,0.816626,3.266505,6.129528,True
3,run4_cyl2,4,2,100,NORCE,2025-07-01,2025-07-15,0.0,14.0,0.009,...,23.307828,NaN,NaN,0.003007,0.012030,6.404447,0.045569,0.182277,6.475390,True
4,run5_cyl1,5,1,60,NORCE,2025-10-23,2025-11-10,0.0,18.0,0.051,...,NaN,NaN,NaN,0.942976,3.771905,6.037139,NaN,NaN,NaN,False
5,run5_cyl2,5,2,60,NORCE,2025-10-23,2025-11-10,0.0,18.0,0.008,...,NaN,NaN,NaN,-0.089438,-0.357751,6.112917,NaN,NaN,NaN,False
6,run6_cyl1,6,1,200,TUD1,2025-11-11,2025-11-24,0.0,13.0,0.082,...,NaN,NaN,NaN,1.005515,4.022061,5.592929,NaN,NaN,NaN,False
7,run6_cyl2,6,2,200,TUD1,2025-11-11,2025-11-24,0.0,13.0,0.001,...,NaN,NaN,NaN,-0.046124,-0.184497,5.323098,NaN,NaN,NaN,False
8,run7_cyl1,7,1,400,TUD1,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
9,run7_cyl2,7,2,400,TUD1,NaT,NaT,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


## 5. Printed report (all columns)
Shows every column of the table. Experiments with data in **both** sulfate columns
have `both_sulfate_data = True`.

In [8]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

print(f"Volume used: {VOLUME_L} L ({VOLUME_L*1000:.1f} mL)   |   H2 : SO4 ratio = {H2_PER_SULFATE} : 1\n")
print(res.to_string(index=False))

Volume used: 0.0675 L (67.5 mL)   |   H2 : SO4 ratio = 4 : 1

        ID  run  cyl  p_bar medium date_start   date_end  day_start  day_end  OD600_start  OD600_end  pH_start  pH_end  acetate_mM_run_start  acetate_mM_run_end  sulfate_mM_run_start  sulfate_mM_run_end  acetate_mM_run1_start  acetate_mM_run1_end  sulfate_mM_run1_start  sulfate_mM_run1_end  Unnamed: 21                   prefix  delta_sulfate_run_mmol  n_hydrogen_run_mmol  n_hydrogen_run_max_mmol  delta_sulfate_run1_mmol  n_hydrogen_run1_mmol  n_hydrogen_run1_max_mmol  both_sulfate_data
 run3_cyl1    3    1    100  NORCE 2025-06-18 2025-06-27        0.0      9.0        0.061      0.021      6.49    8.90                   NaN                 NaN                   NaN                 NaN               2.232385             1.862297              23.031959             9.629919          NaN 2025-06-18_h2-100bar_SRB                     NaN                  NaN                      NaN                 0.904638              3.618551  

## 6. Save results to Excel

In [9]:
res.to_excel(OUTPUT_FILE, index=False)
print(f"Saved -> {OUTPUT_FILE}")

Saved -> /Users/adaciortan/Downloads/new_hydrogen_results.xlsx
